# Week 11 — Smart-City Intelligent Video Analytics

**Pipeline:** Video → YOLO26 → Tracking → ROI/line analytics → JSON events → Incident report → Optional LLM summary

Use a GPU runtime if available.


## 1. Install


In [ ]:
!pip -q install ultralytics==8.4.141 PyYAML yt-dlp


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Upload a short MP4

Preferred: 20–60 seconds, visible people or vehicles, limited camera motion.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from pathlib import Path
mp4s=[p for p in Path('.').glob('*.mp4') if p.name!='input.mp4']
if not Path('input.mp4').exists() and mp4s:
    mp4s[0].rename('input.mp4')
print('input.mp4 exists:',Path('input.mp4').exists())


### Optional public sample source
The following YouTube URL is the example source used in Ultralytics tracking documentation. Skip this if you uploaded your own clip.


In [ ]:
# Optional only:
# !yt-dlp -f "best[height<=720]/best" -o input.mp4 "https://youtu.be/LNwODJXcvt4"


## 3. Inspect first frame


In [ ]:
!python inspect_video.py --source input.mp4 --out outputs/first_frame.jpg
from IPython.display import display
from PIL import Image
display(Image.open('outputs/first_frame.jpg'))


## 4. Inspect `config.yaml`

Adjust normalized ROI/line coordinates if necessary.


In [ ]:
print(open('config.yaml').read())


## 5. Smoke test analytics logic


In [ ]:
!python smoke_test.py


## 6. Quick 200-frame run


In [ ]:
!rm -rf outputs_quick
!python run_demo.py --source input.mp4 --config config.yaml --output-dir outputs_quick --max-frames 200


In [ ]:
import json
print(json.dumps(json.load(open('outputs_quick/metrics.json')),indent=2))


## 7. Full IVA run


In [ ]:
!rm -rf outputs
!mkdir -p outputs
!python run_demo.py --source input.mp4 --config config.yaml --output-dir outputs


## 8. View annotated video


In [ ]:
from IPython.display import Video, display
display(Video('outputs/annotated.mp4',embed=True,width=900))


## 9. Inspect event stream


In [ ]:
from pathlib import Path
events=Path('outputs/events.jsonl').read_text().splitlines()
print('Number of events:',len(events))
for line in events[:15]: print(line)


## 10. Deterministic incident report


In [ ]:
!python summarize_events.py --events outputs/events.jsonl --out-dir outputs


In [ ]:
print(open('outputs/incident_report.txt').read())


## 11. Optional local LLM summary

The LLM sees only structured event facts and is instructed not to invent events.


In [ ]:
!pip -q install "transformers>=4.55" "accelerate>=1.2" sentencepiece


In [ ]:
# Optional:
# !python summarize_events.py --events outputs/events.jsonl --out-dir outputs --llm --llm-model Qwen/Qwen2.5-0.5B-Instruct


## 12. Performance


In [ ]:
import json
m=json.load(open('outputs/metrics.json'))
print('Frames:',m['frames_processed'])
print('Device:',m['device'])
print('Wall FPS:',round(m['wall_fps'],2))
print('Average YOLO inference ms:',None if m['avg_inference_ms'] is None else round(m['avg_inference_ms'],2))
print('Events:',m['event_counts'])


### Discussion
`wall_fps` includes video decoding, YOLO inference, tracking, analytics, drawing, and video encoding. Therefore detector speed alone does not determine application speed.
